# Qwen3-1.7B Reasoning 两阶段微调（SFT → DPO）

**运行环境**：Google Colab（免费版 T4 / Pro A100 均可）

**流程**：安装依赖 → 挂载 Drive → 克隆项目 → SFT 训练 → DPO 训练 → 合并权重 → 推理验证

> ⚠️ 请按顺序逐个 Cell 运行，断线重连后从 **Cell 4（挂载 Drive）** 继续即可。

In [ ]:
# Cell 1：安装依赖（每次新 Session 都需要运行）
!pip install unsloth trl peft bitsandbytes transformers datasets accelerate pyyaml -q
print("✅ 依赖安装完成")

In [ ]:
# Cell 2：设置 HuggingFace Token
# 请先在 Colab 左侧【🔑 Secrets】面板中添加 HF_TOKEN，再运行此 Cell
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("✅ HuggingFace Token 已加载")

In [ ]:
# Cell 3：挂载 Google Drive（防止断线丢失训练结果）
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!mkdir -p "{DRIVE_DIR}/outputs"
print(f"✅ Drive 已挂载，输出目录：{DRIVE_DIR}/outputs")

In [ ]:
# Cell 4：克隆 / 更新项目代码（数据缓存不受影响）
import os

PROJECT_DIR = "/content/6000Q-QwenMiniReason"
REPO_URL    = "https://github.com/yukiiii0730/6000Q-QwenMiniReason.git"

if not os.path.exists(PROJECT_DIR):
    print("📥 首次克隆项目...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("🔄 项目已存在，拉取最新代码...")
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/main
    print("✅ 代码已更新至最新版本")

%cd {PROJECT_DIR}
print("📂 当前目录:", os.getcwd())
!git log --oneline -3

In [ ]:
# Cell 5：检查 GPU 并自动适配配置
import torch
import yaml

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
print(f"GPU : {gpu_name}")
print(f"显存: {vram_gb} GB")

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

with open("config/sft_config.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)

sft_cfg["output_dir"] = f"{DRIVE_DIR}/outputs/sft"
if "T4" in gpu_name:
    sft_cfg["train"]["fp16"] = True
    sft_cfg["train"]["bf16"] = False
    print("⚠️  检测到 T4，已切换为 fp16")
if vram_gb < 20:
    for ds in sft_cfg.get("datasets", []):
        ds["max_samples"] = 10000
    print("⚠️  显存 < 20GB，max_samples 已降至 10000")

with open("config/sft_config.yaml", "w") as f:
    yaml.dump(sft_cfg, f, allow_unicode=True)

with open("config/dpo_config.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)

dpo_cfg["output_dir"]        = f"{DRIVE_DIR}/outputs/dpo"
dpo_cfg["base_adapter_path"] = f"{DRIVE_DIR}/outputs/sft"
if "T4" in gpu_name:
    dpo_cfg["train"]["fp16"] = True
    dpo_cfg["train"]["bf16"] = False
if vram_gb < 20:
    dpo_cfg["dataset"]["max_samples"] = 10000

with open("config/dpo_config.yaml", "w") as f:
    yaml.dump(dpo_cfg, f, allow_unicode=True)

print("✅ 配置已更新完成")

In [ ]:
# Cell 6：数据集下载预检 & 转换为本地缓存（可选但推荐）
# 提前验证三个数据集均可正常拉取，并转换为 Alpaca/DPO 格式存至 Drive
import os
from datasets import load_dataset, concatenate_datasets

DRIVE_DIR   = "/content/drive/MyDrive/Qwen-Reasoning"
PROCESSED   = f"{DRIVE_DIR}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

# ── SFT 数据集 1：NuminaMath-CoT ──────────────────────────────
print("📥 加载 NuminaMath-CoT ...")
numina = load_dataset("AI-MO/NuminaMath-CoT", split="train", trust_remote_code=True)
print(f"   共 {len(numina)} 条，字段: {numina.column_names}")
print(f"   样例 problem : {numina[0]['problem'][:80]}...")
print(f"   样例 solution: {numina[0]['solution'][:80]}...")

# ── SFT 数据集 2：Magpie-Reasoning-150K ───────────────────────
print("\n📥 加载 Magpie-Reasoning-150K ...")
magpie = load_dataset("Magpie-Align/Magpie-Reasoning-150K", split="train", trust_remote_code=True)
print(f"   共 {len(magpie)} 条，字段: {magpie.column_names}")
print(f"   样例 instruction: {magpie[0]['instruction'][:80]}...")
print(f"   样例 response   : {magpie[0]['response'][:80]}...")

# ── DPO 数据集：Orca-Math-Pairs ───────────────────────────────
print("\n📥 加载 Orca-Math-Pairs ...")
orca = load_dataset("microsoft/orca-math-word-problems-200k", split="train", trust_remote_code=True)
print(f"   共 {len(orca)} 条，字段: {orca.column_names}")
print(f"   样例 question          : {orca[0].get('question','N/A')[:80]}...")
print(f"   样例 correct_solution  : {str(orca[0].get('correct_solution','N/A'))[:80]}...")
print(f"   样例 incorrect_solution: {str(orca[0].get('incorrect_solution','N/A'))[:80]}...")

print("\n✅ 三个数据集均可正常访问")

In [ ]:
# Cell 7：将数据集转换并保存到 Drive（加速后续重跑，无需重新下载）
import json, os
from datasets import load_dataset

DRIVE_DIR  = "/content/drive/MyDrive/Qwen-Reasoning"
PROCESSED  = f"{DRIVE_DIR}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

SFT_OUT = f"{PROCESSED}/sft_train.json"
DPO_OUT = f"{PROCESSED}/dpo_train.json"

DEFAULT_INSTRUCTION = "请先进行清晰的逐步推理，再给出最终答案。"

# ============================================================
# ⚡ 快速测试配置（当前值）
#    正式训练建议：NuminaMath 50000、Magpie 50000、Orca 50000
# ============================================================
SFT_NUMINA_N = 500   # 正式建议: 50000
SFT_MAGPIE_N = 500   # 正式建议: 50000
DPO_ORCA_N   = 500   # 正式建议: 50000

# ── 转换 SFT 数据 ─────────────────────────────────────────────
if os.path.exists(SFT_OUT):
    print(f"⏭️  SFT 缓存已存在，跳过转换: {SFT_OUT}")
else:
    print(f"🔄 转换 SFT 数据集（各取 {SFT_NUMINA_N} 条）...")
    sft_rows = []

    numina = load_dataset("AI-MO/NuminaMath-CoT", split="train", trust_remote_code=True).select(range(SFT_NUMINA_N))
    for x in numina:
        sft_rows.append({
            "instruction": DEFAULT_INSTRUCTION,
            "input": x["problem"].strip(),
            "output": x["solution"].strip()
        })

    magpie = load_dataset("Magpie-Align/Magpie-Reasoning-150K", split="train", trust_remote_code=True).select(range(SFT_MAGPIE_N))
    for x in magpie:
        sft_rows.append({
            "instruction": x["instruction"].strip(),
            "input": "",
            "output": x["response"].strip()
        })

    with open(SFT_OUT, "w", encoding="utf-8") as f:
        json.dump(sft_rows, f, ensure_ascii=False, indent=2)
    print(f"✅ SFT 数据已保存：{len(sft_rows)} 条 → {SFT_OUT}")

# ── 转换 DPO 数据 ─────────────────────────────────────────────
if os.path.exists(DPO_OUT):
    print(f"⏭️  DPO 缓存已存在，跳过转换: {DPO_OUT}")
else:
    print(f"🔄 转换 DPO 数据集（取 {DPO_ORCA_N} 条）...")
    orca = load_dataset("microsoft/orca-math-word-problems-200k", split="train", trust_remote_code=True).select(range(DPO_ORCA_N))
    dpo_rows = []
    for x in orca:
        rejected = x.get("incorrect_solution", "")
        if not rejected:
            continue
        dpo_rows.append({
            "prompt":   x.get("question", "").strip(),
            "chosen":   x.get("correct_solution", "").strip(),
            "rejected": rejected.strip()
        })
    with open(DPO_OUT, "w", encoding="utf-8") as f:
        json.dump(dpo_rows, f, ensure_ascii=False, indent=2)
    print(f"✅ DPO 数据已保存：{len(dpo_rows)} 条 → {DPO_OUT}")


# ── 把本地缓存路径写回 config（保留 Cell 5 的 fp16/bf16 设置）─────────
import yaml

with open("config/sft_config.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)
sft_cfg.pop("datasets", None)
sft_cfg["dataset_path"] = SFT_OUT
# ⚡ 快速测试: max_steps=50；正式建议: 1000
sft_cfg["train"]["max_steps"] = 50   # 正式建议: 1000
# 保留 Cell 5 写入的 fp16/bf16（不覆盖，避免 T4 bf16 报错）
import torch as _t
if not (_t.cuda.is_available() and _t.cuda.is_bf16_supported()):
    sft_cfg["train"]["bf16"] = False
    sft_cfg["train"]["fp16"] = True
with open("config/sft_config.yaml", "w") as f:
    yaml.dump(sft_cfg, f, allow_unicode=True)

with open("config/dpo_config.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)
dpo_cfg.pop("dataset", None)
dpo_cfg["dataset_path"] = DPO_OUT
# ⚡ 快速测试: max_steps=30；正式建议: 800
dpo_cfg["train"]["max_steps"] = 30   # 正式建议: 800
if not (_t.cuda.is_available() and _t.cuda.is_bf16_supported()):
    dpo_cfg["train"]["bf16"] = False
    dpo_cfg["train"]["fp16"] = True
with open("config/dpo_config.yaml", "w") as f:
    yaml.dump(dpo_cfg, f, allow_unicode=True)

print("\n✅ 配置已切换为本地缓存模式（快速测试参数）")

In [ ]:
# Cell 8：基础模型 GSM8K 基线测试（微调前，用于对比）
import re, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

DRIVE_DIR       = "/content/drive/MyDrive/Qwen-Reasoning"
BASE_MODEL      = "Qwen/Qwen3-1.7B"
BASELINE_OUTPUT = f"{DRIVE_DIR}/outputs/gsm8k_baseline.json"

# ============================================================
# ⚡ 快速测试: GSM8K_N=50；正式建议: 1319（完整测试集）
# ============================================================
GSM8K_N = 50   # 正式建议: 1319

def extract_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else ""

print(f"📥 加载基础模型 {BASE_MODEL} ...")
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto",
    torch_dtype=torch.float16, trust_remote_code=True
)

print(f"📊 加载 GSM8K 测试集（前 {GSM8K_N} 条）...")
gsm = load_dataset("gsm8k", "main", split="test").select(range(GSM8K_N))

correct, details = 0, []
for i, ex in enumerate(gsm):
    prompt = f"请解答以下数学题，逐步推理后在最后一行写出数字答案。\n题目：{ex['question']}\n解答："
    inputs  = tok(prompt, return_tensors="pt").to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=256, do_sample=False)
    pred_text = tok.decode(outputs[0], skip_special_tokens=True)
    pred_num  = extract_number(pred_text)
    gt_num    = extract_number(ex["answer"])
    ok = (pred_num == gt_num and pred_num != "")
    correct += int(ok)
    details.append({"question": ex["question"], "pred": pred_num, "gt": gt_num, "correct": ok})
    if (i + 1) % 10 == 0:
        print(f"  进度 {i+1}/{GSM8K_N}，当前准确率 {correct/(i+1):.2%}")

acc = correct / max(len(details), 1)
result = {"model": BASE_MODEL, "accuracy": acc, "total": len(details),
          "correct": correct, "details": details}

with open(BASELINE_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"\n✅ 基础模型 GSM8K 基线：{acc:.2%}  ({correct}/{len(details)})")
print(f"   结果已保存至: {BASELINE_OUTPUT}")

In [ ]:
# Cell 9：第一阶段 SFT 训练
!python scripts/sft_train.py --config config/sft_config.yaml

In [ ]:
# Cell 10：验证 SFT 产物是否已保存到 Drive
import os
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
print("SFT 输出文件:", os.listdir(f"{DRIVE_DIR}/outputs/sft"))

In [ ]:
# Cell 11：第二阶段 DPO 训练
!python scripts/dpo_train.py --config config/dpo_config.yaml

In [ ]:
# Cell 12：合并 LoRA 权重到基础模型
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!python scripts/merge_lora.py \
    --adapter_path "{DRIVE_DIR}/outputs/dpo" \
    --output_path  "{DRIVE_DIR}/outputs/merged"
print("✅ 权重合并完成")

In [ ]:
# Cell 13：推理验证——对比基础模型 vs 微调模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

DRIVE_DIR  = "/content/drive/MyDrive/Qwen-Reasoning"
BASE_MODEL = "Qwen/Qwen3-1.7B"
FT_MODEL   = f"{DRIVE_DIR}/outputs/merged"
QUESTION   = "小明有12个苹果，给了小红3个，又买了5个，现在有几个？请一步步推理后给出答案。"

def infer(model_path, prompt, max_new_tokens=512):
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
    )
    inputs  = tok(prompt, return_tensors="pt").to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("【基础模型】")
print(infer(BASE_MODEL, QUESTION))
print()
print("=" * 60)
print("【微调模型（SFT + DPO）】")
print(infer(FT_MODEL, QUESTION))

In [ ]:
# Cell 14（可选）：GSM8K 微调后评测（与 Cell 8 基线对比）
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!python eval/gsm8k_eval.py \
    --model_path  "{DRIVE_DIR}/outputs/merged" \
    --max_samples 200 \
    --output      eval/gsm8k_result.json

import json
with open("eval/gsm8k_result.json") as f:
    result = json.load(f)
print(f"GSM8K Accuracy (微调后): {result['accuracy']:.2%}  ({result['correct']}/{result['total']})")

# 与基线对比
try:
    with open(f"{DRIVE_DIR}/outputs/gsm8k_baseline.json") as f:
        baseline = json.load(f)
    print(f"GSM8K Accuracy (基线):   {baseline['accuracy']:.2%}  ({baseline['correct']}/{baseline['total']})")
    delta = result['accuracy'] - baseline['accuracy']
    print(f"提升幅度: {delta:+.2%}")
except FileNotFoundError:
    print("（未找到基线文件，跳过对比）")